# Optimal Execution with Regime Uncertainty: Methods Comparison

Comparing four control approaches:
1. **REINFORCE** - Deep reinforcement learning
2. **Naive** - Belief-weighted mean of regime-specific solutions
3. **Certainty Equivalent** - Expected parameters approach  
4. **Oracle** - Perfect information upper bound

**Problem**: Optimal execution with regime uncertainty, maximizing expected profit:

$$\max_{u} \mathbb{E}\left[\int_0^T \left( (Y_s - \rho u_s) u_s - c X_s^2 \right) ds + Y_T X_T - C X_T^2\right]$$

**Dynamics**:
- Price: $dY_s = -f(\lambda_\xi, \kappa_\xi, u_s, \alpha_s^\xi) ds + \sigma dW_s$
- Inventory: $dX_s = -u_s ds$, $X_0 = x_0$
- Hidden regime: $\xi_s \in \{L, H\}$ (unobservable)
- Resilience: $d\alpha_s^{\xi} = (u_s + \kappa_\xi \alpha_s^{\xi}) ds$

**Analytical Solutions**: All classical methods use Riccati-based optimal control with linear feedback law:
$$u^*(s) = v_0(T-s) \cdot [v_1(T-s) \cdot X(s) + v_2(T-s) \cdot \alpha(s)]$$

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import matplotlib.pyplot as plt
import warnings
import yaml
from typing import Optional, NamedTuple, Dict, Any, List, Tuple, Union, Callable, Protocol
from dataclasses import dataclass
import flax.linen as nn
import optax
from abc import ABC, abstractmethod

warnings.filterwarnings('ignore')

# Type aliases for better readability
State = jnp.ndarray  # Shape: (5,) representing [Y, X, p, alpha_l, alpha_h]
Action = Union[float, jnp.ndarray]  # Single action value
NetworkParams = Dict[str, Any]  # Flax model parameters
TrajectoryDict = Dict[str, Union[jnp.ndarray, List[Dict[str, Any]], float, int]]
MetricsDict = Dict[str, float]
ResultsDict = Dict[str, Any]
TrainingHistory = Dict[str, List[float]]

@dataclass
class OptimalExecutionConfig:
    """Configuration for optimal execution problem."""
    lambda_l: float = 0.5
    kappa_l: float = 10.0
    lambda_h: float = 2.0
    kappa_h: float = 2.0
    rho: float = 0.1
    sigma: float = 0.2
    c: float = 0.01
    C: float = 10.0
    T: float = 1.0
    Y_0: float = 100.0
    X_0: float = 10.0
    p_0: float = 0.5
    dt: float = 0.01
    n_steps: Optional[int] = None
    
    def __post_init__(self) -> None:
        if self.n_steps is None:
            self.n_steps = int(self.T / self.dt)
    
    @property
    def time_grid(self) -> jnp.ndarray:
        return jnp.linspace(0, self.T, self.n_steps + 1)
    
    @property
    def regime_params(self) -> Tuple[float, float, float, float]:
        return (self.lambda_l, self.kappa_l, self.lambda_h, self.kappa_h)
    
    @property
    def initial_state(self) -> State:
        return jnp.array([self.Y_0, self.X_0, self.p_0, 0.0, 0.0])

default_config = OptimalExecutionConfig()

class PolicyProtocol(Protocol):
    """Protocol defining the interface for trading policies."""
    
    def __call__(self, state: State, time: float) -> Action:
        """Execute policy given current state and time."""
        ...
    
    @property
    def name(self) -> str:
        """Return policy name for identification."""
        ...

class Policy(ABC):
    """Abstract base class for trading policies."""
    
    @abstractmethod
    def __call__(self, state: State, time: float) -> Action:
        """Execute policy given current state and time."""
        pass
    
    @property
    def name(self) -> str:
        return self.__class__.__name__

def compute_riccati_action(
    state: State, 
    time: float, 
    config: OptimalExecutionConfig, 
    kappa: float, 
    alpha: float
) -> Action:
    """Unified Riccati-based optimal control computation.
    
    Implements the linear feedback law: u*(s) = v0(τ) * [v1(τ) * X + v2(τ) * α]
    where τ = T - s is time-to-go.
    
    Args:
        state: Current state [Y, X, p, alpha_l, alpha_h]
        time: Current time
        config: Problem configuration
        kappa: Resilience parameter to use
        alpha: Alpha value to use
        
    Returns:
        Optimal action (trading rate)
    """
    Y, X, p, alpha_l, alpha_h = state
    tau = jnp.maximum(config.T - time, 1e-6)
    
    base = 1.0 + tau / (2.0 * config.rho)
    v0 = 1.0 / base
    v1 = (config.C / config.rho) / (base ** 2)
    v2 = 2.0 * kappa * config.rho * v0 * (1.0 - v0)
    
    action = v0 * (v1 * X + v2 * alpha)
    return jnp.clip(action, 0.0, X / config.dt)

class OraclePolicy(Policy):
    """Oracle policy with perfect regime knowledge."""
    
    def __init__(self, config: OptimalExecutionConfig = default_config) -> None:
        self.config = config
        self.true_regime: Optional[float] = None
    
    def set_true_regime(self, regime: float) -> None:
        """Set the true regime for oracle policy."""
        self.true_regime = regime
    
    def __call__(self, state: State, time: float) -> Action:
        if self.true_regime is None:
            self.true_regime = 0.0
        Y, X, p, alpha_l, alpha_h = state
        kappa = (1 - self.true_regime) * self.config.kappa_l + self.true_regime * self.config.kappa_h
        alpha = (1 - self.true_regime) * alpha_l + self.true_regime * alpha_h
        return compute_riccati_action(state, time, self.config, kappa, alpha)
    
    @property
    def name(self) -> str:
        return "Oracle"

class CertaintyEquivalentPolicy(Policy):
    """Certainty equivalent policy using expected parameters."""
    
    def __init__(self, config: OptimalExecutionConfig = default_config) -> None:
        self.config = config
    
    def __call__(self, state: State, time: float) -> Action:
        Y, X, p, alpha_l, alpha_h = state
        kappa = p * self.config.kappa_l + (1 - p) * self.config.kappa_h
        alpha = p * alpha_l + (1 - p) * alpha_h
        return compute_riccati_action(state, time, self.config, kappa, alpha)
    
    @property
    def name(self) -> str:
        return "Certainty Equivalent"

class NaivePolicy(Policy):
    """Naive policy averaging regime-specific optimal controls."""
    
    def __init__(self, config: OptimalExecutionConfig = default_config) -> None:
        self.config = config
    
    def __call__(self, state: State, time: float) -> Action:
        Y, X, p, alpha_l, alpha_h = state
        action_l = compute_riccati_action(state, time, self.config, self.config.kappa_l, alpha_l)
        action_h = compute_riccati_action(state, time, self.config, self.config.kappa_h, alpha_h)
        return p * action_l + (1 - p) * action_h
    
    @property
    def name(self) -> str:
        return "Naive"

@dataclass
class REINFORCEConfig:
    """Configuration for REINFORCE algorithm training."""
    n_episodes: int = 10000
    hidden_dim: int = 64
    learning_rate: float = 3e-4
    batch_size: int = 10
    log_interval: int = 100
    gamma: float = 0.99
    baseline_alpha: float = 0.01
    
class PolicyNetwork(nn.Module):
    """Neural network for REINFORCE policy."""
    hidden_dim: int = 64
    log_std_min: float = -5.0
    log_std_max: float = 2.0
    
    @nn.compact
    def __call__(self, x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """Forward pass returning mean and log_std of action distribution."""
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.tanh(x)
        mean = nn.Dense(1)(x)
        log_std = nn.Dense(1)(x)
        log_std = jnp.clip(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std

class REINFORCEPolicy(Policy):
    """REINFORCE-based policy using neural networks."""
    
    def __init__(
        self, 
        network: nn.Module, 
        params: NetworkParams, 
        config: OptimalExecutionConfig = default_config
    ) -> None:
        self.network = network
        self.params = params
        self.config = config
        self._forward = jax.jit(self.network.apply)
    
    def __call__(
        self, 
        state: State, 
        time: float, 
        key: Optional[random.PRNGKey] = None
    ) -> Action:
        """Execute policy with optional stochasticity."""
        network_input = jnp.concatenate([state, jnp.array([time])])
        mean, log_std = self._forward({"params": self.params}, network_input)
        mean, log_std = mean.squeeze(), log_std.squeeze()
        
        if key is not None:
            std = jnp.exp(log_std)
            action = mean + std * random.normal(key)
        else:
            action = mean
        
        Y, X, p, alpha_l, alpha_h = state
        action = jnp.maximum(action, 0.0)
        action = jnp.minimum(action, X / self.config.dt)
        return action
    
    @property
    def name(self) -> str:
        return "REINFORCE"

class REINFORCEAgent:
    """REINFORCE training agent."""
    
    def __init__(
        self, 
        config: OptimalExecutionConfig = default_config, 
        reinforce_config: Optional[REINFORCEConfig] = None
    ) -> None:
        self.config = config
        self.reinforce_config = reinforce_config or REINFORCEConfig()
        self.network = PolicyNetwork(hidden_dim=self.reinforce_config.hidden_dim)
        self.params: Optional[NetworkParams] = None
        self.optimizer: Optional[optax.GradientTransformation] = None
        self.opt_state: Optional[optax.OptState] = None
        self.baseline: float = 0.0
        self.env: Optional['OptimalExecutionEnv'] = None
    
    def _init_params(self, key: random.PRNGKey) -> None:
        """Initialize network parameters and optimizer."""
        dummy_input = jnp.ones(6)
        self.params = self.network.init(key, dummy_input)["params"]
        self.optimizer = optax.adam(self.reinforce_config.learning_rate)
        self.opt_state = self.optimizer.init(self.params)
    
    def _sample_action(
        self, 
        state: State, 
        time: float, 
        params: NetworkParams, 
        key: random.PRNGKey
    ) -> Tuple[Action, float]:
        """Sample action and compute log probability."""
        network_input = jnp.concatenate([state, jnp.array([time])])
        mean, log_std = self.network.apply({"params": params}, network_input)
        mean, log_std = mean.squeeze(), log_std.squeeze()
        std = jnp.exp(log_std)
        action = mean + std * random.normal(key)
        Y, X, p, alpha_l, alpha_h = state
        action = jnp.maximum(action, 0.0)
        action = jnp.minimum(action, X / self.config.dt)
        log_prob = -0.5 * jnp.log(2 * jnp.pi) - log_std - 0.5 * ((action - mean) / std) ** 2
        return action, log_prob
    
    def _run_episode(
        self, 
        params: NetworkParams, 
        key: random.PRNGKey
    ) -> Tuple[jnp.ndarray, jnp.ndarray, float]:
        """Run a single training episode."""
        state = self.env.reset(key)
        states, actions, rewards, log_probs = [], [], [], []
        
        for t in range(self.config.n_steps):
            current_time = t * self.config.dt
            key, action_key = random.split(key)
            action, log_prob = self._sample_action(state, current_time, params, action_key)
            
            states.append(state)
            actions.append(action)
            log_probs.append(log_prob)
            
            key, step_key = random.split(key)
            result = self.env.step(state, action, step_key)
            state = result.next_state
            rewards.append(result.reward)
            
            if result.done:
                break
        
        terminal_reward = self.env.compute_terminal_reward(state)
        rewards.append(terminal_reward)
        
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.reinforce_config.gamma * G
            returns.insert(0, G)
        
        returns = jnp.array(returns[:-1])
        log_probs = jnp.array(log_probs)
        
        return returns, log_probs, jnp.sum(jnp.array(rewards))
    
    def train(self, key: random.PRNGKey) -> TrainingHistory:
        """Train the REINFORCE agent."""
        key, init_key = random.split(key)
        self._init_params(init_key)
        
        history: TrainingHistory = {'rewards': [], 'losses': []}
        
        for episode in range(self.reinforce_config.n_episodes):
            key, episode_key = random.split(key)
            returns, log_probs, total_reward = self._run_episode(self.params, episode_key)
            
            self.baseline = (1 - self.reinforce_config.baseline_alpha) * self.baseline + \
                           self.reinforce_config.baseline_alpha * jnp.mean(returns)
            
            advantages = returns - self.baseline
            loss = -jnp.mean(log_probs * advantages)
            
            grads = jax.grad(lambda p: -jnp.mean(
                jnp.array([self._sample_action(s, t * self.config.dt, p, k)[1] for s, t, k in 
                          zip(jnp.zeros((len(log_probs), 5)), jnp.arange(len(log_probs)), 
                              random.split(episode_key, len(log_probs)))]) * advantages
            ))(self.params)
            
            updates, self.opt_state = self.optimizer.update(grads, self.opt_state, self.params)
            self.params = optax.apply_updates(self.params, updates)
            
            history['rewards'].append(float(total_reward))
            history['losses'].append(float(loss))
            
            if (episode + 1) % self.reinforce_config.log_interval == 0:
                avg_reward = np.mean(history['rewards'][-self.reinforce_config.log_interval:])
                print(f"Episode {episode + 1}/{self.reinforce_config.n_episodes}, Avg Reward: {avg_reward:.2f}")
        
        return history
    
    def create_policy(self) -> REINFORCEPolicy:
        """Create a policy from trained parameters."""
        if self.params is None:
            raise ValueError("Agent must be trained before creating policy")
        return REINFORCEPolicy(self.network, self.params, self.config)

In [ ]:
class StepResult(NamedTuple):
    next_state: State
    reward: float
    done: bool
    info: Dict[str, Any]

class OptimalExecutionEnv:
    """Environment for optimal execution simulation."""
    
    def __init__(self, config: OptimalExecutionConfig = default_config) -> None:
        self.config = config
        self.true_regime: Optional[float] = None
        self.sqrt_dt = jnp.sqrt(config.dt)
        
        self._step_fn = jax.jit(self._step_impl)
        self._batch_step_fn = jax.vmap(self._step_impl, in_axes=(0, 0, 0, None))
    
    def reset(self, key: random.PRNGKey, batch_size: int = 1) -> Union[State, jnp.ndarray]:
        """Reset environment to initial state."""
        if batch_size == 1:
            self.true_regime = random.bernoulli(key, self.config.p_0).astype(jnp.float32)
            return self.config.initial_state
        else:
            keys = random.split(key, batch_size)
            self.true_regime = random.bernoulli(keys, self.config.p_0).astype(jnp.float32)
            return jnp.tile(self.config.initial_state, (batch_size, 1))
    
    def step(self, state: State, action: Action, key: random.PRNGKey) -> StepResult:
        """Take a single step in the environment."""
        return self._step_fn(state, action, key, self.true_regime)
    
    def batch_step(self, states: jnp.ndarray, actions: jnp.ndarray, key: random.PRNGKey) -> StepResult:
        """Take a batch of steps in the environment."""
        keys = random.split(key, states.shape[0])
        return self._batch_step_fn(states, actions, keys, self.true_regime)
    
    def _step_impl(
        self, 
        state: State, 
        action: Action, 
        key: random.PRNGKey, 
        true_regime: float
    ) -> StepResult:
        """Internal step implementation."""
        Y, X, p, alpha_l, alpha_h = state
        u = action
        
        lambda_l, kappa_l, lambda_h, kappa_h = self.config.regime_params
        
        true_lambda = (1 - true_regime) * lambda_l + true_regime * lambda_h
        true_kappa = (1 - true_regime) * kappa_l + true_regime * kappa_h
        true_alpha = (1 - true_regime) * alpha_l + true_regime * alpha_h
        
        true_drift = -true_lambda * (u + true_kappa * true_alpha)
        
        dW = random.normal(key) * self.sqrt_dt
        dY = true_drift * self.config.dt + self.config.sigma * dW
        
        expected_alpha = p * alpha_l + (1 - p) * alpha_h
        expected_lambda = p * lambda_l + (1 - p) * lambda_h
        expected_kappa = p * kappa_l + (1 - p) * kappa_h
        expected_drift = -expected_lambda * (u + expected_kappa * expected_alpha)
        
        innovation = dY - expected_drift * self.config.dt
        
        f_low = lambda_l * (u + kappa_l * alpha_l)
        f_high = lambda_h * (u + kappa_h * alpha_h)
        drift_difference = f_low - f_high
        
        dp = (1 / (self.config.sigma**2)) * p * (1 - p) * drift_difference * innovation * self.config.dt
        
        Y_next = Y + dY
        X_next = X - u * self.config.dt
        p_next = jnp.clip(p + dp, 0.0, 1.0)
        alpha_l_next = alpha_l + (u + kappa_l * alpha_l) * self.config.dt
        alpha_h_next = alpha_h + (u + kappa_h * alpha_h) * self.config.dt
        
        next_state = jnp.array([Y_next, X_next, p_next, alpha_l_next, alpha_h_next])
        
        execution_revenue = (Y - self.config.rho * u) * u * self.config.dt
        inventory_cost = -self.config.c * X**2 * self.config.dt
        reward = execution_revenue + inventory_cost
        
        done = jnp.abs(X_next) < 1e-6
        
        info = {
            'true_regime': true_regime,
            'belief': p_next,
            'price': Y_next,
            'inventory': X_next,
            'execution_revenue': execution_revenue,
            'inventory_cost': inventory_cost,
            'resilience_l': alpha_l_next,
            'resilience_h': alpha_h_next,
            'innovation': innovation
        }
        
        return StepResult(next_state, reward, done, info)
    
    def compute_terminal_reward(self, state: State) -> float:
        """Compute terminal reward for given state."""
        Y, X, _, _, _ = state
        return Y * X - self.config.C * X**2
    
    def generate_trajectory(
        self, 
        policy_fn: Callable[[State, float], Action], 
        n_steps: Optional[int] = None, 
        key: random.PRNGKey = random.PRNGKey(0)
    ) -> TrajectoryDict:
        """Generate a complete trajectory using the given policy."""
        if n_steps is None:
            n_steps = self.config.n_steps
        
        key, reset_key = random.split(key)
        state = self.reset(reset_key)
        
        states = jnp.zeros((n_steps + 1, 5))
        actions = jnp.zeros(n_steps) 
        rewards = jnp.zeros(n_steps)
        infos = []
        
        states = states.at[0].set(state)
        
        for t in range(n_steps):
            current_time = t * self.config.dt
            action = policy_fn(state, current_time)
            actions = actions.at[t].set(action)
            
            key, step_key = random.split(key)
            result = self.step(state, action, step_key)
            
            state = result.next_state
            states = states.at[t + 1].set(state)
            rewards = rewards.at[t].set(result.reward)
            infos.append(result.info)
            
            if result.done:
                break
        
        terminal_reward = self.compute_terminal_reward(state)
        
        return {
            'states': states,
            'actions': actions,
            'rewards': rewards,
            'terminal_reward': terminal_reward,
            'total_reward': jnp.sum(rewards) + terminal_reward,
            'infos': infos,
            'final_state': state,
            'n_steps': n_steps
        }

In [ ]:
@dataclass
class PolicyResult:
    """Results from policy evaluation."""
    name: str
    total_rewards: jnp.ndarray
    final_inventories: jnp.ndarray
    trajectories: List[TrajectoryDict]
    metrics: MetricsDict

class OptimalExecutionComparator:
    """Comparator for different optimal execution methods."""
    
    def __init__(
        self, 
        config: OptimalExecutionConfig = default_config,
        reinforce_config: Optional[REINFORCEConfig] = None
    ) -> None:
        self.config = config
        self.env = OptimalExecutionEnv(config)
        self.reinforce_config = reinforce_config or REINFORCEConfig(n_episodes=10000)
    
    def compare_all_methods(
        self, 
        key: random.PRNGKey = random.PRNGKey(42),
        n_evaluation_episodes: int = 500,
        verbose: bool = True
    ) -> ResultsDict:
        """Compare all control methods and return detailed results."""
        results = {}
        keys = random.split(key, 4)
        
        if verbose:
            print("Training REINFORCE and evaluating all control methods...")
        
        if verbose:
            print("Training REINFORCE...")
        reinforce_policy, learning_history = self._train_reinforce_with_curves(keys[0], verbose)
        results['reinforce'] = self._evaluate_policy_detailed(
            reinforce_policy, keys[0], n_evaluation_episodes
        )
        results['reinforce']['method'] = 'REINFORCE'
        results['reinforce']['learning_history'] = learning_history
        
        if verbose:
            print("Evaluating Certainty Equivalent...")
        ce_policy = CertaintyEquivalentPolicy(self.config)
        results['certainty_equivalent'] = self._evaluate_policy_detailed(
            ce_policy, keys[1], n_evaluation_episodes
        )
        results['certainty_equivalent']['method'] = 'Certainty Equivalent'
        
        if verbose:
            print("Evaluating Naive...")
        naive_policy = NaivePolicy(self.config)
        results['naive'] = self._evaluate_policy_detailed(
            naive_policy, keys[2], n_evaluation_episodes
        )
        results['naive']['method'] = 'Naive'
        
        if verbose:
            print("Evaluating Oracle...")
        oracle_policy = OraclePolicy(self.config)
        results['oracle'] = self._evaluate_oracle_detailed(
            oracle_policy, keys[3], n_evaluation_episodes
        )
        results['oracle']['method'] = 'Oracle'
        
        if verbose:
            self._print_summary(results)
        
        return results
    
    def _train_reinforce_with_curves(
        self, 
        key: random.PRNGKey, 
        verbose: bool = True
    ) -> Tuple[REINFORCEPolicy, TrainingHistory]:
        """Train REINFORCE agent and return policy with learning curves."""
        agent = REINFORCEAgent(self.config, self.reinforce_config)
        agent.env = OptimalExecutionEnv(self.config)
        history = agent.train(key)
        return agent.create_policy(), history
    
    def _evaluate_policy_detailed(
        self, 
        policy: Policy, 
        key: random.PRNGKey, 
        n_episodes: int
    ) -> Dict[str, Any]:
        """Evaluate policy with detailed metrics collection."""
        episode_keys = random.split(key, n_episodes)
        
        total_rewards = []
        final_inventories = []
        trajectories = []
        
        for i, episode_key in enumerate(episode_keys):
            trajectory = self.env.generate_trajectory(policy, key=episode_key)
            trajectories.append(trajectory)
            
            total_rewards.append(float(trajectory['total_reward']))
            final_inventories.append(float(trajectory['final_state'][1]))
            
            if (i + 1) % 100 == 0 and i > 0:
                print(f"  {i + 1}/{n_episodes} episodes")
        
        total_rewards = jnp.array(total_rewards)
        final_inventories = jnp.array(final_inventories)
        
        return {
            'total_rewards': total_rewards,
            'final_inventories': final_inventories,
            'trajectories': trajectories[:10],
            'metrics': self._compute_detailed_metrics(total_rewards, final_inventories)
        }
    
    def _evaluate_oracle_detailed(
        self, 
        oracle_policy: OraclePolicy, 
        key: random.PRNGKey,
        n_episodes: int
    ) -> Dict[str, Any]:
        """Evaluate oracle policy with perfect regime knowledge."""
        episode_keys = random.split(key, n_episodes)
        
        total_rewards = []
        final_inventories = []
        trajectories = []
        
        for i, episode_key in enumerate(episode_keys):
            env_key, traj_key = random.split(episode_key)
            self.env.reset(env_key)
            oracle_policy.set_true_regime(self.env.true_regime)
            
            trajectory = self.env.generate_trajectory(oracle_policy, key=traj_key)
            trajectories.append(trajectory)
            
            total_rewards.append(float(trajectory['total_reward']))
            final_inventories.append(float(trajectory['final_state'][1]))
            
            if (i + 1) % 100 == 0 and i > 0:
                print(f"  {i + 1}/{n_episodes} episodes")
        
        total_rewards = jnp.array(total_rewards)
        final_inventories = jnp.array(final_inventories)
        
        return {
            'total_rewards': total_rewards,
            'final_inventories': final_inventories,
            'trajectories': trajectories[:10],
            'metrics': self._compute_detailed_metrics(total_rewards, final_inventories)
        }
    
    def _compute_detailed_metrics(
        self, 
        total_rewards: jnp.ndarray, 
        final_inventories: jnp.ndarray
    ) -> MetricsDict:
        """Compute comprehensive performance metrics."""
        return {
            'mean_reward': float(jnp.mean(total_rewards)),
            'std_reward': float(jnp.std(total_rewards)),
            'min_reward': float(jnp.min(total_rewards)),
            'max_reward': float(jnp.max(total_rewards)),
            'median_reward': float(jnp.median(total_rewards)),
            'reward_q25': float(jnp.percentile(total_rewards, 25)),
            'reward_q75': float(jnp.percentile(total_rewards, 75)),
            'mean_final_inventory': float(jnp.mean(jnp.abs(final_inventories))),
            'std_final_inventory': float(jnp.std(final_inventories)),
            'liquidation_rate': float(jnp.mean(jnp.abs(final_inventories) < 0.1)),
            'sharpe_ratio': float(jnp.mean(total_rewards) / (jnp.std(total_rewards) + 1e-8)),
        }
    
    def _print_summary(self, results: ResultsDict) -> None:
        """Print performance summary table."""
        print("\\nPerformance Summary:")
        print(f"{'Method':<20} {'Mean Reward':<12} {'Liquidation':<12}")
        for method_name, result in results.items():
            metrics = result['metrics']
            print(f"{result['method']:<20} "
                  f"{metrics['mean_reward']:<12.1f} "
                  f"{metrics['liquidation_rate']:<12.1%}")
        
        ranking = sorted(results.items(), 
                        key=lambda x: x[1]['metrics']['mean_reward'], 
                        reverse=True)
        print("\\nRanking:")
        for i, (method_name, result) in enumerate(ranking):
            print(f"{i+1}. {result['method']}: {result['metrics']['mean_reward']:.1f}")
    
    def plot_comprehensive_results(
        self, 
        results: ResultsDict, 
        save_path: Optional[str] = None, 
        show: bool = True
    ) -> None:
        """Generate comprehensive visualization of results."""
        fig = plt.figure(figsize=(20, 12))
        
        # Performance comparison bar chart
        ax1 = plt.subplot(2, 2, 1)
        methods = []
        means = []
        stds = []
        
        for method_name, result in results.items():
            methods.append(result['method'])
            means.append(result['metrics']['mean_reward'])
            stds.append(result['metrics']['std_reward'])
        
        colors = ['red', 'blue', 'green', 'orange'][:len(methods)]
        bars = ax1.bar(methods, means, yerr=stds, capsize=5, color=colors, alpha=0.7)
        ax1.set_ylabel('Mean Total Reward')
        ax1.set_title('Performance Comparison')
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # REINFORCE learning curve (if available)
        ax2 = plt.subplot(2, 2, 2)
        if 'reinforce' in results and 'learning_history' in results['reinforce']:
            history = results['reinforce']['learning_history']
            episodes = np.arange(len(history['rewards']))
            
            window_size = min(100, len(history['rewards']) // 10)
            window_size = max(1, window_size)
            
            ax2.plot(episodes, history['rewards'], alpha=0.3, color='blue', label='Raw')
            
            if window_size > 1 and len(history['rewards']) > window_size:
                smoothed_rewards = np.convolve(history['rewards'], 
                                             np.ones(window_size)/window_size, 
                                             mode='valid')
                smoothed_episodes = episodes[window_size-1:]
                ax2.plot(smoothed_episodes, smoothed_rewards, color='blue', linewidth=2, label='Smoothed')
            ax2.set_xlabel('Episode')
            ax2.set_ylabel('Total Reward')
            ax2.set_title('REINFORCE Learning Curve')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
        else:
            ax2.axis('off')
            ax2.text(0.5, 0.5, 'No REINFORCE learning history available', 
                    ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        
        # Sample price trajectories
        ax3 = plt.subplot(2, 2, 3)
        time_grid = self.config.time_grid
        
        for i, (method_name, result) in enumerate(results.items()):
            if 'trajectories' in result and result['trajectories']:
                traj = result['trajectories'][0]
                prices = traj['states'][:, 0]
                ax3.plot(time_grid[:len(prices)], prices, 
                        label=result['method'], color=colors[i % len(colors)], linewidth=2)
        
        ax3.set_xlabel('Time')
        ax3.set_ylabel('Price')
        ax3.set_title('Sample Price Trajectories')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Sample inventory trajectories  
        ax4 = plt.subplot(2, 2, 4)
        
        for i, (method_name, result) in enumerate(results.items()):
            if 'trajectories' in result and result['trajectories']:
                traj = result['trajectories'][0]
                inventory = traj['states'][:, 1]
                ax4.plot(time_grid[:len(inventory)], inventory,
                        label=result['method'], color=colors[i % len(colors)], linewidth=2)
        
        ax4.set_xlabel('Time')
        ax4.set_ylabel('Inventory')
        ax4.set_title('Sample Inventory Trajectories')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
        if show:
            plt.show()

In [ ]:
with open('src/model_parameters_test.yaml', 'r') as f:
    params = yaml.safe_load(f)

config = OptimalExecutionConfig(
    T=params['T'],
    dt=params['DT'], 
    lambda_l=params['LAMBDA_L'],
    lambda_h=params['LAMBDA_H'],  
    kappa_l=params['KAPPA_L'],
    kappa_h=params['KAPPA_H'],
    rho=params['RHO'],
    sigma=params['SIGMA'],
    c=params['C_RUNNING'],
    C=params['C_TERMINAL'],
    Y_0=params['Y_0'],
    X_0=params['X_0'],
    p_0=params['P_0']
)

reinforce_config = REINFORCEConfig(
    n_episodes=params['REINFORCE']['n_episodes'],
    hidden_dim=params['REINFORCE']['hidden_dim'],
    learning_rate=params['REINFORCE']['learning_rate'],
    batch_size=params['REINFORCE']['batch_size'],
    log_interval=params['REINFORCE']['log_interval']
)

comparator = OptimalExecutionComparator(config, reinforce_config)

print(f"Configuration loaded: Liquidate {config.X_0} shares over {config.T} time units")

In [ ]:
key = random.PRNGKey(42)
results = comparator.compare_all_methods(
    key=key, 
    n_evaluation_episodes=params['COMPARISON']['n_evaluation_episodes'],
    verbose=True
)

print("Comparison completed. Results ready for analysis.")

In [ ]:
print("Performance Summary:")
for method_name, result in results.items():
    metrics = result['metrics']
    print(f"{result['method']}: Mean Reward = {metrics['mean_reward']:.1f}, "
          f"Liquidation = {metrics['liquidation_rate']:.1%}")

ranking = sorted(results.items(), key=lambda x: x[1]['metrics']['mean_reward'], reverse=True)
print("\nRanking:")
for i, (method_name, result) in enumerate(ranking):
    print(f"{i+1}. {result['method']}: {result['metrics']['mean_reward']:.1f}")

In [ ]:
print("Detailed Performance Metrics:")
for method_name, result in results.items():
    metrics = result['metrics']
    print(f"\n{result['method']}:")
    print(f"  Reward: {metrics['mean_reward']:.1f} ± {metrics['std_reward']:.1f}")
    print(f"  Sharpe: {metrics['sharpe_ratio']:.2f}")
    print(f"  Liquidation: {metrics['liquidation_rate']:.1%}")
    
    if 'learning_history' in result:
        history = result['learning_history']
        print(f"  Training Episodes: {len(history['rewards'])}")
        print(f"  Best Episode: {np.max(history['rewards']):.1f}")

print("\nAll analytical methods use Riccati-based optimal control solutions")
print("Oracle provides theoretical upper bound with perfect regime knowledge")

In [ ]:
print("Generating comprehensive visualization...")
comparator.plot_comprehensive_results(results, show=True)